# DeepSORT
This notebook tests the DeepSORT model

## Imports

In [ ]:
import glob
import sys
from pathlib import Path
import os
import cv2
from google.colab.patches import cv2_imshow

In [ ]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from src.utils.config.config import Config
from src.utils.draw_boxes import DrawBoxes
from src.detector.yolo_detector import YOLODetector
from src.tracker.deepsort_tracker import DeepSortTracker

## Config

In [ ]:
config_loader = Config()
cfg = config_loader.load_config()

## Test function

In [ ]:
def test_deepsort(detector, tracker, output_path, dataset_path, save_video=True, play_frames=False):
  # Config
  # save_video = True # Creates a video from all the frames
  # play_frames = False # Plays the frames while processing
  frames_path = sorted(glob.glob(dataset_path + "/test/images/*.jpg"))
  output_video_path = str(output_path / "deepsort_output.mp4")
  
  draw_boxes = DrawBoxes()
  
  id_counts = set() # Hold the IDs that were tracked
  track_lengths = {} # Holds the amount of frames each id was alive for

  # Validating the frames path
  if not frames_path:
      raise RuntimeError("No frames found")

  # Ensure the output directory exists
  os.makedirs(output_path, exist_ok=True)

  # Retrieve first frame dimensions
  if save_video:
    first_frame = cv2.imread(frames_path[0])
    height, width = first_frame.shape[:2]
    fps = 5

    # Define the codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 files
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

  # Loop through images/frames
  for img_path in frames_path:

    # Read the frame
    frame = cv2.imread(img_path)
    if frame is None:
      continue

    # Process the frame
    processed_frame = frame.copy()
    
    # Run through YOLO
    yolo_results = detector.predict(frame)
    if yolo_results is None:
      continue
    
    # Convert to deepsort format and sort detections
    human_dets, object_dets = tracker.yolo_to_deepsort_split(yolo_results)
    
    # Run through DeepSORT (only tracks humans)
    deepsort_results = tracker.update(human_dets, frame)

    # Draw DeepSORT results
    for track in deepsort_results:
      if not track.is_confirmed():
          continue
    
      track_id = int(track.track_id)
      bbox = track.to_ltrb()
      class_name = track.get_det_class()
      
      id_counts.add(track_id)
      track_lengths[track_id] = track_lengths.get(track_id, 0) + 1
      
      # frame, bboxes, class_names, track_ids
      processed_frame = draw_boxes.draw_box(processed_frame, bbox, class_name, track_id=track_id)

    # Draw the non-human objects
    for bbox,_,class_name,class_id in object_dets:
      processed_frame = draw_boxes.draw_box(processed_frame, bbox, class_name, class_id=class_id)

    if save_video: out.write(processed_frame) # Write current frame to output
    if play_frames: cv2_imshow(processed_frame) # output current frame

    # if cv2.waitKey(1) & 0xFF == ord('q'):
      # break

  # Release resources
  if save_video: out.release()
  # cv2.destroyAllWindows()
  
  return id_counts, track_lengths

## Run Test

### Init YOLO

In [ ]:
detector = YOLODetector(cfg.yolo.model_path, cfg.project_root)

### Init DeepSORT

In [ ]:
tracker = DeepSortTracker(cfg.deepsort.max_age,
                          cfg.deepsort.n_init,
                          cfg.deepsort.nn_budget,
                          cfg.deepsort.max_cosine_distance,
                          cfg.deepsort.max_iou_distance,
                          cfg.deepsort.nms_max_overlap
                        )

### Temporary Dataset

In [ ]:
dataset_path = ""

### Run

In [ ]:
track_ids, track_lengths = test_deepsort(detector, 
              tracker, 
              cfg.paths.output_videos, 
              dataset_path,
              save_video=True,
              play_frames=False
            )

print("Total unique tracks:", len(track_ids))
print("Average track length:", sum(track_lengths.values()) / len(track_lengths))